In [1]:
!pip install spacy sentence-transformers PyPDF2 python-docx requests
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 8.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 103.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [23]:
import os
import re
import PyPDF2
import docx
import requests
import spacy
from google.colab import files
from sentence_transformers import SentenceTransformer, util
from typing import List, Dict

# Load models
nlp = spacy.load("en_core_web_sm")
similarity_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

def extract_text_from_file(file_path: str) -> str:
    """
    Extract text from various file formats (PDF, DOCX, TXT)
    """
    file_ext = os.path.splitext(file_path)[1].lower()

    if file_ext == '.pdf':
        with open(file_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            text = ""
            for page in pdf_reader.pages:
                text += page.extract_text() + "\n"
        return text

    elif file_ext == '.docx':
        doc = docx.Document(file_path)
        text = "\n".join([para.text for para in doc.paragraphs])
        return text

    elif file_ext == '.txt':
        with open(file_path, 'r', encoding='utf-8') as file:
            return file.read()

    else:
        raise ValueError(f"Unsupported file format: {file_ext}")

print("Upload a PDF, DOCX, or TXT file:")
uploaded = files.upload()

# Get the uploaded filename
file_path = next(iter(uploaded.keys()))

# Extract text
document_text = extract_text_from_file(file_path)

print(f"\n✅ Extracted {len(document_text)} characters from file")
print(f"Preview: {document_text[:200]}...\n")


Upload a PDF, DOCX, or TXT file:


Saving infosec_a2.pdf to infosec_a2 (3).pdf

✅ Extracted 13795 characters from file
Preview: MalGuard: Graph-based Detection of Malicious PyPI Packages
Final Report — Information Security (CS-3002) — Assignment 2
Abdul Wasay (22i-2037) Manhab Zafar (22i-1957)
November 2025
Abstract
In this re...



In [25]:
def extract_sentences(text: str) -> List[str]:
    """Extract meaningful sentences using spaCy"""
    doc = nlp(text)
    sentences = [sent.text.strip() for sent in doc.sents]
    # Filter sentences with at least 8 words
    return [s for s in sentences if len(s.split()) >= 8]

def extract_key_info(sentence: str) -> Dict:
    """Extract entities and key phrases from a sentence"""
    doc = nlp(sentence)

    entities = [(ent.text, ent.label_) for ent in doc.ents]
    noun_chunks = [chunk.text for chunk in doc.noun_chunks if len(chunk.text.split()) >= 2]

    return {
        'sentence': sentence,
        'entities': entities,
        'concepts': noun_chunks
    }

def generate_flashcards_local(text: str, max_cards: int = 10) -> List[Dict]:
    """
    Generate flashcards using local spaCy model
    Creates Q/A pairs based on entities and concepts
    """
    sentences = extract_sentences(text)
    flashcards = []

    for sentence in sentences[:max_cards * 2]:  # Process more to filter best
        info = extract_key_info(sentence)

        # Create flashcards from entities
        for entity, label in info['entities']:
            if label in ['PERSON', 'ORG', 'GPE', 'DATE', 'EVENT']:
                question = sentence.replace(entity, "______")
                flashcard = {
                    'question': f"Fill in the blank: {question}",
                    'answer': entity,
                    'context': sentence,
                    'type': 'entity',
                    'confidence': 0.8
                }
                flashcards.append(flashcard)

        # Create definition-style flashcards from concepts
        if info['concepts']:
            main_concept = info['concepts'][0]
            flashcard = {
                'question': f"What is {main_concept}?",
                'answer': sentence,
                'context': sentence,
                'type': 'concept',
                'confidence': 0.7
            }
            flashcards.append(flashcard)

    # Use MiniLM to rank by importance (diversity)
    if len(flashcards) > max_cards:
        embeddings = similarity_model.encode([f['context'] for f in flashcards])
        # Simple diversity sampling: pick distributed cards
        selected_indices = range(0, len(flashcards), len(flashcards) // max_cards)
        flashcards = [flashcards[i] for i in selected_indices][:max_cards]

    return flashcards

# Generate NEW flashcards from the uploaded document
print("🔄 Generating flashcards using LOCAL MODEL (spaCy + MiniLM)...\n")
local_flashcards = generate_flashcards_local(document_text, max_cards=10)

print(f"✅ Generated {len(local_flashcards)} flashcards\n")
for i, card in enumerate(local_flashcards, 1):
    print(f"--- FLASHCARD {i} ({card['type']}) ---")
    print(f"Q: {card['question']}")
    print(f"A: {card['answer']}")
    print(f"Confidence: {card['confidence']}\n")


🔄 Generating flashcards using LOCAL MODEL (spaCy + MiniLM)...

✅ Generated 10 flashcards

--- FLASHCARD 1 (concept) ---
Q: What is Graph-based Detection?
A: MalGuard: Graph-based Detection of Malicious PyPI Packages
Final Report — Information Security (CS-3002) —
Confidence: 0.7

--- FLASHCARD 2 (entity) ---
Q: Fill in the blank: Assignment 2
Abdul Wasay (22i-2037) Manhab Zafar (______)
November 2025
A: 22i-1957
Confidence: 0.8

--- FLASHCARD 3 (entity) ---
Q: Fill in the blank: In this report, we provide details of a system to identify malicious Python packages in
the PyPI ecosystem based on the ______-call graph analysis and machine learning (MalGuard).
A: API
Confidence: 0.8

--- FLASHCARD 4 (entity) ---
Q: Fill in the blank: It is based on previous research on the threats of ______ and suggests a complete pipeline:
package extraction, API-call graph construction, extraction of centrality-based features, top-
K selection of API and classification using a variety of machine learning 

In [26]:
import time

def interactive_flashcard_session(flashcards: List[Dict]):
    """
    Interactive flashcard study session with quiz mode
    """
    if not flashcards:
        print("❌ No flashcards available!")
        return

    print("=" * 60)
    print("🎓 INTERACTIVE FLASHCARD STUDY SESSION")
    print("=" * 60)
    print(f"\nTotal flashcards: {len(flashcards)}")
    print("\nModes:")
    print("1. Review Mode - See Q&A one by one")
    print("2. Quiz Mode - Test yourself")
    print("3. Shuffle & Quiz")

    mode = input("\nSelect mode (1/2/3): ").strip()

    if mode == "1":
        review_mode(flashcards)
    elif mode == "2":
        quiz_mode(flashcards)
    elif mode == "3":
        import random
        random.shuffle(flashcards)
        quiz_mode(flashcards)
    else:
        print("Invalid choice!")

def review_mode(flashcards: List[Dict]):
    """Review flashcards one by one"""
    print("\n" + "=" * 60)
    print("📖 REVIEW MODE")
    print("=" * 60)

    for i, card in enumerate(flashcards, 1):
        print(f"\n{'=' * 60}")
        print(f"FLASHCARD {i}/{len(flashcards)}")
        print(f"{'=' * 60}\n")
        print(f"❓ QUESTION:\n{card['question']}\n")

        input("Press Enter to reveal answer...")

        print(f"\n✅ ANSWER:\n{card['answer']}\n")

        if i < len(flashcards):
            next_action = input("\n[N]ext card, [Q]uit: ").strip().lower()
            if next_action == 'q':
                break

    print("\n✨ Review session complete!")

def quiz_mode(flashcards: List[Dict]):
    """Quiz mode - test knowledge with automatic grading"""
    print("\n" + "=" * 60)
    print("🎯 QUIZ MODE")
    print("=" * 60)

    correct = 0
    incorrect = 0
    skipped = 0
    results = []  # Store detailed results

    for i, card in enumerate(flashcards, 1):
        # Show progress WITHOUT clearing
        print(f"\n{'=' * 60}")
        print(f"QUESTION {i}/{len(flashcards)}")
        print(f"Score: ✅ {correct} | ❌ {incorrect} | ⏭️ {skipped}")
        print(f"{'=' * 60}\n")
        print(f"❓ {card['question']}\n")

        user_answer = input("Your answer (or type 'skip' to skip): ").strip()

        if user_answer.lower() == 'skip' or user_answer == '':
            skipped += 1
            print(f"\n⏭️ Skipped. Correct answer was: {card['answer']}\n")
            results.append({
                'question': card['question'],
                'user_answer': 'SKIPPED',
                'correct_answer': card['answer'],
                'status': 'skipped'
            })
            continue

        # Automatic grading using similarity
        is_correct = check_answer_similarity(user_answer, card['answer'])

        if is_correct:
            correct += 1
            print(f"\n✅ CORRECT!")
            print(f"Your answer: {user_answer}")
            print(f"Expected: {card['answer']}")
            print("🎉 Great job!\n")
            results.append({
                'question': card['question'],
                'user_answer': user_answer,
                'correct_answer': card['answer'],
                'status': 'correct'
            })
        else:
            incorrect += 1
            print(f"\n❌ INCORRECT")
            print(f"Your answer: {user_answer}")
            print(f"Correct answer: {card['answer']}")
            print("💪 Keep practicing!\n")
            results.append({
                'question': card['question'],
                'user_answer': user_answer,
                'correct_answer': card['answer'],
                'status': 'incorrect'
            })

    # Final results
    total = len(flashcards)
    accuracy = (correct / total * 100) if total > 0 else 0

    print("\n" + "=" * 60)
    print("📊 DETAILED QUIZ RESULTS")
    print("=" * 60)
    print(f"\n✅ Correct: {correct}/{total}")
    print(f"❌ Incorrect: {incorrect}/{total}")
    print(f"⏭️ Skipped: {skipped}/{total}")
    print(f"\n🎯 Accuracy: {accuracy:.1f}%")

    # Show incorrect answers for review
    if incorrect > 0:
        print("\n" + "=" * 60)
        print("📝 REVIEW INCORRECT ANSWERS")
        print("=" * 60)
        for idx, result in enumerate([r for r in results if r['status'] == 'incorrect'], 1):
            print(f"\n{idx}. Question: {result['question']}")
            print(f"   Your answer: {result['user_answer']}")
            print(f"   Correct answer: {result['correct_answer']}")

    if accuracy >= 80:
        print("\n🌟 Excellent! You've mastered this material!")
    elif accuracy >= 60:
        print("\n👍 Good job! Review the missed ones.")
    else:
        print("\n📚 Keep studying! You'll get there!")

    return results

def check_answer_similarity(user_answer: str, correct_answer: str, threshold: float = 0.65) -> bool:
    """
    Check if user's answer is similar enough to the correct answer
    Uses multiple methods: exact match, fuzzy match, and semantic similarity
    """
    user_lower = user_answer.lower().strip()
    correct_lower = correct_answer.lower().strip()

    # Method 1: Exact match (case-insensitive)
    if user_lower == correct_lower:
        return True

    # Method 2: Check if answer is contained in correct answer or vice versa
    if user_lower in correct_lower or correct_lower in user_lower:
        # If one is significantly shorter, check overlap percentage
        shorter = min(len(user_lower), len(correct_lower))
        longer = max(len(user_lower), len(correct_lower))
        if shorter / longer > 0.5:  # At least 50% overlap
            return True

    # Method 3: Fuzzy string matching (Levenshtein-like)
    similarity_ratio = fuzzy_match(user_lower, correct_lower)
    if similarity_ratio > threshold:
        return True

    # Method 4: Semantic similarity using MiniLM
    try:
        user_embedding = similarity_model.encode(user_answer, convert_to_tensor=True)
        correct_embedding = similarity_model.encode(correct_answer, convert_to_tensor=True)
        semantic_score = util.cos_sim(user_embedding, correct_embedding).item()

        if semantic_score > threshold:
            return True
    except:
        pass  # If model fails, rely on string methods

    return False

def fuzzy_match(s1: str, s2: str) -> float:
    """
    Calculate similarity ratio between two strings (simple implementation)
    Returns value between 0 and 1
    """
    # Tokenize and compare word overlap
    words1 = set(s1.split())
    words2 = set(s2.split())

    if not words1 or not words2:
        return 0.0

    intersection = words1.intersection(words2)
    union = words1.union(words2)

    return len(intersection) / len(union) if union else 0.0

def export_flashcards(flashcards: List[Dict], filename: str = "flashcards.txt"):
    """Export flashcards to a text file"""
    with open(filename, 'w', encoding='utf-8') as f:
        f.write("FLASHCARD DECK\n")
        f.write("=" * 60 + "\n\n")

        for i, card in enumerate(flashcards, 1):
            f.write(f"CARD {i}\n")
            f.write(f"Q: {card['question']}\n")
            f.write(f"A: {card['answer']}\n")
            f.write(f"Context: {card.get('context', 'N/A')}\n")
            f.write("-" * 60 + "\n\n")

    print(f"✅ Flashcards exported to {filename}")

# Use the freshly generated flashcards
study_cards = local_flashcards
print(f"\n📚 Using {len(study_cards)} local model flashcards for study session")

# Start interactive session
interactive_flashcard_session(study_cards)

# Option to export
export_choice = input("\n\nWould you like to export these flashcards? (y/n): ").strip().lower()
if export_choice == 'y':
    filename = input("Enter filename (default: flashcards.txt): ").strip() or "flashcards.txt"
    export_flashcards(study_cards, filename)


📚 Using 10 local model flashcards for study session
🎓 INTERACTIVE FLASHCARD STUDY SESSION

Total flashcards: 10

Modes:
1. Review Mode - See Q&A one by one
2. Quiz Mode - Test yourself
3. Shuffle & Quiz

Select mode (1/2/3): 3

🎯 QUIZ MODE

QUESTION 1/10
Score: ✅ 0 | ❌ 0 | ⏭️ 0

❓ What is 7
6 Results 7
6.1 Confusion matrices?

Your answer (or type 'skip' to skip): true positiives

❌ INCORRECT
Your answer: true positiives
Correct answer: 7
6 Results 7
6.1 Confusion matrices (summary) . . . . . . . . . . . . . . . . . . . . . . . . . . . .
💪 Keep practicing!


QUESTION 2/10
Score: ✅ 0 | ❌ 1 | ⏭️ 0

❓ Fill in the blank: In this report, we provide details of a system to identify malicious Python packages in
the PyPI ecosystem based on the ______-call graph analysis and machine learning (MalGuard).

Your answer (or type 'skip' to skip): API

✅ CORRECT!
Your answer: API
Expected: API
🎉 Great job!


QUESTION 3/10
Score: ✅ 1 | ❌ 1 | ⏭️ 0

❓ Fill in the blank: Assignment 2
Abdul Wasay (22i-203